In [0]:
from pyspark.sql import functions as F

In [0]:
df_gold = spark.table("workspace.silver.alunos_tratados")


In [0]:
df_gold.printSchema()

### Quantidade de alunos por município

In [0]:
gold_municipios = (
    df_gold.filter(F.col("STATUS_MUNICIPIO") == "VALIDO")
        .groupBy("MUNICIPIO")
        .agg(F.count("*").alias("TOTAL_ALUNOS"))
        .orderBy(F.desc("TOTAL_ALUNOS"))
)

In [0]:
# Persistência da tabela

gold_municipios.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.alunos_por_municipio")

### Quantidade de alunos por curso

In [0]:
gold_alunos_curso = (
    df_gold.groupBy("NOME_CURSO")
        .agg(F.count("*").alias("TOTAL_ALUNOS"))
        .orderBy(F.desc("TOTAL_ALUNOS"))
)

In [0]:
# Persistência da tabela

gold_alunos_curso.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.alunos_por_curso")

### Quantidade de alunos por estado

In [0]:
gold_alunos_estado = df_gold.filter(
        (F.col("UF") != "XX") &
        (F.col("UF").isNotNull())
    ) \
    .groupBy("UF") \
    .agg(F.count("*").alias("TOTAL_ALUNOS")) \
    .orderBy(F.desc("TOTAL_ALUNOS"))

In [0]:
# Persistência da tabela

gold_alunos_estado.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.alunos_por_estado")

### Quais municípios enviam mais alunos para cada curso

In [0]:
gold_alunos_por_municipio_curso = (
    df_gold
    .filter(F.col("STATUS_MUNICIPIO") == "VALIDO")
    .groupBy("MUNICIPIO", "NOME_CURSO")
    .agg(
        F.count("*").alias("TOTAL_ALUNOS")
    )
)

In [0]:
from pyspark.sql.window import Window

In [0]:
from pyspark.sql.functions import row_number

#ordenando a quantidade de alunos
janela = Window.partitionBy("NOME_CURSO") \
    .orderBy(F.desc("TOTAL_ALUNOS"))


In [0]:
#aplicando a ordenação e criando coluna de rank
gold_top_municipio_por_curso = (
    gold_alunos_por_municipio_curso
    .withColumn("RANK", row_number().over(janela))
    .filter(F.col("RANK") == 1)
    .drop("RANK")
)

In [0]:
# Persistência da tabela

gold_top_municipio_por_curso.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.top_municipio_por_curso")

### Quantidade de alunos de cada ano

In [0]:
gold_alunos_ano = (
    df_gold.withColumn(
        "ANO_INGRESSO",
        F.substring("MATR_ALUNO", 1, 4)
    )
    .groupBy("ANO_INGRESSO")
    .agg(
        F.count("*").alias("TOTAL_ALUNOS")
    )
    .orderBy("ANO_INGRESSO")
)

In [0]:
# Persistência da tebala
(
    gold_alunos_ano.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.alunos_por_ano")
)

### Quantidade de alunos ingressantes de 2025 por curso

In [0]:
gold_alunos_2025_curso = (
    df_gold
    .withColumn(
        "ANO_INGRESSO",
        F.substring("MATR_ALUNO", 1, 4)
    )
    .filter(F.col("ANO_INGRESSO") == "2025")
    .groupBy("NOME_CURSO")
    .agg(
        F.count("*").alias("TOTAL_ALUNOS")
    )
    .orderBy(F.desc("TOTAL_ALUNOS"))
)

In [0]:
# Persistência da tabela

gold_alunos_2025_curso.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.alunos_2025_curso")